# Topic: Data Leakage

## Definition (30-second explanation)
*   Data leakage occurs when information from outside the training dataset is inadvertently used to create the model.
*   This causes artificially inflated performance metrics during evaluation but leads to complete failure in the real world (production).

## Why Interviewers Ask This
*   It is one of the most common and dangerous mistakes in machine learning.
*   It directly impacts a model's ability to generalize, turning a "perfect" model into a useless business asset.

## Core Concepts
*   **Preprocessing / Train-Test Leakage**: Fitting transformers (e.g., `StandardScaler`, PCA) on the full dataset before splitting, allowing test data statistics to influence training.
*   **Target Leakage**: Including future information that leaks into features (e.g., using "loan repaid" to predict default).
*   **Temporal Leakage**: Using future time period data to predict past events instead of using proper chronological splits.
*   **Duplicate Leakage**: Having the exact same records present in both the training and test sets.

## When to Use (Prevention Strategies)
*   Always split your data *before* applying any preprocessing or feature engineering.
*   Use `sklearn.pipeline.Pipeline` to safely chain preprocessing and modeling.
*   Always use temporal splits (never random splits) for time-series data.
*   For every feature, ask: "Would this feature actually be available at prediction time in production?".

## How to Detect
*   Metrics are suspiciously high (e.g., accuracy > 99%, AUC > 0.99) on held-out test data.
*   A single feature exhibits dramatically higher importance than all other features combined.
*   The model performs substantially worse on new production data compared to the test set.

## Common Interview Traps
*   **The Scaler Trap**: Using `train_test_split` *after* calling `StandardScaler.fit_transform(X)` is the most common leakage mistake. The scaler uses the mean and standard deviation from the test set, meaning the model indirectly "sees" test data.
*   **Insidious Target Leakage**: Target leakage features often disguise themselves as highly predictive, becoming the most important feature in the model. Suspiciously high test accuracy is always a red flag that requires immediate investigation.

## Python / SQL Syntax 
```python
# CORRECT: Pipeline prevents leakage
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

pipeline = Pipeline([
    ('scaler', StandardScaler()), # Fit ONLY on train
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train) # Scaler fit here on train only
acc_good = accuracy_score(y_test, pipeline.predict(X_test))
```

## 45-Second Interview Answer
"Data leakage is when outside information—usually from the test set or future events—accidentally contaminates the training process. It leads to artificially inflated metrics, like 99% accuracy during validation, but terrible performance in production. The two most common types are preprocessing leakage, where we fit a scaler on the whole dataset before splitting, and target leakage, where we include a feature that wouldn't actually be available at prediction time. To prevent this, I always split my data before any transformation, use scikit-learn Pipelines to encapsulate preprocessing, and rigorously question if every feature is truly known at the moment of prediction."

## Example Questions:

**Question:** Your model has 99.5% accuracy on the test set but only 60% accuracy in production. What is the most likely cause?

**Ideal Interview Answer:** 
This is a classic sign of data leakage. The most likely causes are: 
1. **Preprocessing leakage:** The test set was included when fitting a preprocessor like a scaler. 
2. **Target leakage:** A feature that is only available after the prediction target is realized was included in the training data.
3. **Duplicate records:** Duplicates existed across the train and test sets. 
To fix this, I would audit the feature engineering pipeline, ensure strict temporal splits if applicable, and enforce the use of `sklearn.pipeline.Pipeline` for all preprocessing to guarantee transformations are only fit on the training split.

**Common Mistakes Candidates Make:**
Failing to specifically mention the exact mechanism of the error, such as applying `train_test_split` *after* `StandardScaler.fit_transform`, which allows the model to indirectly 'see' the test data via the mean and standard deviation.

**Likely Interviewer Follow-up:**
"If you suspect target leakage, how would you go about identifying exactly which feature is causing it during your EDA or model evaluation phase?"

## Practice Questions:

### Q1:
**Question:** 
You are building a model to predict loan defaults. On your validation set, your Random Forest achieves 0.99 AUC. The feature `late_payment_fee_total` has an 88% relative importance score. What is happening, why, and how do you fix it?

**Ideal Interview Answer:** 
This is a textbook case of **Target Leakage**. The feature `late_payment_fee_total` is highly predictive because late fees are only accumulated *after* a customer begins to default. Because this information would not be known at the time of loan origination (the moment of prediction in production), it is leaking future information into the model. 

To resolve this, I would:
1. Immediately remove `late_payment_fee_total` from the training data.
2. Conduct a thorough audit of the remaining features, asking "Would this exact value be definitively known at the exact moment we need to run this prediction in production?" for every single feature.

**Interview Tip:**
Always explicitly use the term "Target Leakage." When a model performs suspiciously well (e.g., > 0.95 AUC on tabular business data) or a single feature dominates importance, immediately state that your first instinct is to investigate for target leakage.

### Q2:
# Interview Scenario: Identifying Preprocessing Leakage in Code

**Question:** 
Review this code. Is there data leakage? Identify where, explain mathematically why, and provide the corrected code.
```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Assume df is loaded
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate
print("Accuracy:", model.score(X_test, y_test))
```

**Answer:** 
Yes, there is **Preprocessing Leakage** (or Train-Test Leakage). 
The issue occurs here: `X_scaled = scaler.fit_transform(X)` *before* the data is split.

**Mathematically why it fails:** 
`StandardScaler` calculates the mean ($\mu$) and standard deviation ($\sigma$) of the dataset to compute $z = \frac{x - \mu}{\sigma}$. By fitting it on the entire `X`, the mean and variance of the *test set* are baked into the scaling of the *training set*. The model indirectly "sees" the test distribution during training, inflating evaluation metrics.

**Corrected Code:**
```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Split FIRST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

# 3. Fit pipeline only on training data
pipeline.fit(X_train, y_train)

# 4. Evaluate
print("Accuracy:", pipeline.score(X_test, y_test))
```

**Interview Tip:**
Always advocate for sklearn.pipeline.Pipeline. It proves to the interviewer that you know how to write production-safe, leak-proof machine learning code, rather than just academic scripts.